In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import mne
import numpy as np
import pandas as pd
import warnings
from contextlib import contextmanager
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler, label_binarize
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns

warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================

CONTROL_FOLDER = "/kaggle/input/my-dataset/Dataset/Control EEG"
CONTROL_FOLDER_2 = ""

DS_FOLDER = "/kaggle/input/new-ds-data-v2/SODIUM CHANNEL"
DS_FOLDER_2 = ""

ABNORMAL_FOLDER = "/kaggle/input/abnormal-eegv1/Abnormal"

FS = 250
WINDOW_SEC = 20
OVERLAP = 0.5
OUTPUT_DIR = "/kaggle/working"
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.0001


ARTIFACT_KEYWORDS = [
    'impedance', 'deleted eeg', 'deleted video', 'movement artifact', 'crying',
    'head movement', 'patient keep moving', 'patient not cooperative', 'uncooperative',
    'restless', 'fixing electrode', 'fix electrode', 'calibration', 'start recording', 'stop recording',
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ============================================================
# UTILITY FUNCTIONS
# ============================================================

@contextmanager
def suppress_stdout():
    import sys
    from io import StringIO
    old_stdout = sys.stdout
    sys.stdout = StringIO()
    try:
        yield
    finally:
        sys.stdout = old_stdout


def detect_artifact_regions(raw, artifact_keywords=None):
    if artifact_keywords is None:
        artifact_keywords = ARTIFACT_KEYWORDS
    
    artifact_regions = []
    
    if raw.annotations is not None:
        for a in raw.annotations:
            description = a['description'].lower()
            is_artifact = any(keyword in description for keyword in artifact_keywords)
            
            if is_artifact:
                artifact_regions.append({
                    'start_time': a['onset'],
                    'end_time': a['onset'] + a['duration'],
                    'description': a['description']
                })
    
    return artifact_regions


def remove_artifact_chunks(chunks, artifact_regions, fs, window_sec, overlap):
    chunk_size = int(window_sec * fs)
    step = int(chunk_size * (1 - overlap))
    step = max(step, 1)
    
    filtered_chunks = []
    removed_count = 0
    
    for idx, chunk in enumerate(chunks):
        start_time = (idx * step) / fs
        end_time = start_time + window_sec
        
        is_bad = False
        for artifact in artifact_regions:
            if start_time < artifact['end_time'] and end_time > artifact['start_time']:
                is_bad = True
                removed_count += 1
                break
        
        if not is_bad:
            filtered_chunks.append(chunk)
    
    return filtered_chunks, removed_count


def load_eeg_folder(folder_path, label, fs, window_sec, overlap, remove_artifacts=True):
    channels = [
        'EEG F3-Cz','EEG Fz-Cz','EEG F4-Cz',
        'EEG C3-Cz','EEG C4-Cz',
        'EEG P3-Cz','EEG Pz-Cz','EEG P4-Cz'
    ]

    chunk_size = int(window_sec*fs)
    step = int(chunk_size*(1-overlap))
    step = max(step,1)

    all_chunks=[]; all_labels=[]; all_files=[]
    total_removed = 0
    skipped_count = 0

    for f in tqdm(os.listdir(folder_path), desc=f"Loading {os.path.basename(folder_path)}"):
        if not f.endswith('.edf'): 
            continue
        
    
        path = os.path.join(folder_path, f)
        try:
            with suppress_stdout():
                raw = mne.io.read_raw_edf(path, preload=True)
                raw.pick_channels(channels)
            
            artifact_regions = detect_artifact_regions(raw) if remove_artifacts else []
            
            eeg = raw.get_data()
            n = eeg.shape[1]

            chunks=[]
            for st in range(0, n-chunk_size+1, step):
                en = st + chunk_size
                chunks.append(eeg[:, st:en].astype(np.float32))

            if remove_artifacts and len(artifact_regions) > 0:
                chunks, removed = remove_artifact_chunks(chunks, artifact_regions, fs, window_sec, overlap)
                total_removed += removed

            all_chunks.extend(chunks)
            all_labels.extend([label]*len(chunks))
            all_files.extend([f]*len(chunks))

        except Exception as e:
            print(f"Error loading {f}: {str(e)}")

  
    if remove_artifacts and total_removed > 0:
        print(f"   → Removed {total_removed} artifact-contaminated chunks")

    return all_chunks, all_labels, all_files


def load_eeg_multi_folders(folder_paths, label, fs, window_sec, overlap, remove_artifacts=True):
    all_chunks = []
    all_labels = []
    all_files = []
    loaded_subjects = set()
    
    for folder_path in folder_paths:
        if not os.path.exists(folder_path):
            print(f"   ⚠ Folder not found: {folder_path}")
            continue
        
        chunks, labels, files = load_eeg_folder(folder_path, label, fs, window_sec, overlap, remove_artifacts)
        all_chunks.extend(chunks)
        all_labels.extend(labels)
        all_files.extend(files)
        loaded_subjects.update(files)
    
   
      
      
        if missing:
            print(f"   ⚠ Missing: {', '.join(sorted(missing))}")
    
    return all_chunks, all_labels, all_files

# ============================================================
# PYTORCH DATASET
# ============================================================

class EEGDataset(Dataset):
    def __init__(self, data, labels, subjects):
        self.data = data
        self.labels = labels
        self.subjects = subjects
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        x = torch.FloatTensor(self.data[idx])
        y = torch.LongTensor([self.labels[idx]])[0]
        return x, y


# ============================================================
# CUSTOM CNN-LSTM MODEL (NON-PRETRAINED)
# ============================================================

class CustomCNNLSTM(nn.Module):
    """Custom CNN + LSTM hybrid model for EEG classification"""
    def __init__(self, n_channels=8, n_classes=3, n_times=5000, lstm_hidden=64):
        super(CustomCNNLSTM, self).__init__()
        
        # CNN blocks for spatial feature extraction
        self.cnn = nn.Sequential(
            # Conv block 1
            nn.Conv1d(n_channels, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(0.3),
            
            # Conv block 2
            nn.Conv1d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(0.3),
            
            # Conv block 3
            nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Dropout(0.3),
        )
        
        # LSTM layers for temporal pattern extraction
        self.lstm_input_size = 256
        self.lstm_seq_len = n_times // 8
        
        self.lstm = nn.LSTM(
            input_size=self.lstm_input_size,
            hidden_size=lstm_hidden,
            num_layers=2,
            batch_first=True,
            dropout=0.4
        )
        
        # Fully connected layers for classification
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, n_classes)
        )
    
    def forward(self, x):
        # x shape: (batch_size, n_channels, n_times)
        
        # CNN feature extraction
        cnn_out = self.cnn(x)  # (batch_size, 256, seq_len)
        
        # Reshape for LSTM: (batch_size, seq_len, 256)
        cnn_out = cnn_out.permute(0, 2, 1)
        
        # LSTM temporal modeling
        lstm_out, (h_n, c_n) = self.lstm(cnn_out)  # lstm_out: (batch_size, seq_len, lstm_hidden)
        
        # Use last LSTM output
        lstm_last = lstm_out[:, -1, :]  # (batch_size, lstm_hidden)
        
        # Classification
        out = self.fc(lstm_last)  # (batch_size, n_classes)
        return out


def get_model(n_channels=8, n_classes=3, n_times=5000, lstm_hidden=64):
    """Create custom CNN-LSTM model"""
    model = CustomCNNLSTM(n_channels=n_channels, n_classes=n_classes, n_times=n_times, lstm_hidden=lstm_hidden)
    return model


# ============================================================
# TRAINING FUNCTIONS
# ============================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += batch_y.size(0)
        correct += predicted.eq(batch_y).sum().item()
    
    return total_loss / len(loader), 100. * correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    
    return total_loss / len(loader), acc * 100, f1, all_labels, all_preds


# ============================================================
# METRICS CALCULATION
# ============================================================

def calculate_metrics(y_true, y_pred, class_names=['Control', 'DS', 'Abnormal']):
    """Calculate accuracy, balanced accuracy, F1, sensitivity, specificity, precision, NPV"""
    
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }
    
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    
    for i, class_name in enumerate(class_names):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - tp - fp - fn
        
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0
        
        metrics[f'{class_name}_sensitivity'] = sensitivity
        metrics[f'{class_name}_specificity'] = specificity
        metrics[f'{class_name}_precision'] = precision
        metrics[f'{class_name}_npv'] = npv
    
    return metrics


def calculate_roc_auc(y_true, y_proba, class_names):
    """Calculate ROC-AUC for multi-class (One-vs-Rest)"""
    
    y_bin = label_binarize(y_true, classes=[0, 1, 2])
    
    roc_scores = {}
    try:
        roc_auc_macro = roc_auc_score(y_bin, y_proba, multi_class='ovr', average='macro')
        roc_scores['roc_auc_macro'] = roc_auc_macro
        
        roc_auc_weighted = roc_auc_score(y_bin, y_proba, multi_class='ovr', average='weighted')
        roc_scores['roc_auc_weighted'] = roc_auc_weighted
        
        for i, class_name in enumerate(class_names):
            y_bin_i = y_bin[:, i]
            roc_auc_i = roc_auc_score(y_bin_i, y_proba[:, i])
            roc_scores[f'roc_auc_{class_name}'] = roc_auc_i
    except:
        roc_scores['roc_auc_macro'] = 0
        roc_scores['roc_auc_weighted'] = 0
        for class_name in class_names:
            roc_scores[f'roc_auc_{class_name}'] = 0
    
    return roc_scores


# ============================================================
# LOSO CROSS-VALIDATION
# ============================================================

def run_loso_cv(df, model_name='CustomCNNLSTM'):
    """Leave-One-Subject-Out Cross-Validation with pooled predictions"""
    
    subjects = df['file_name'].unique()
    results = []
    pooled_y_true = []
    pooled_y_pred = []
    pooled_y_proba = []
    class_names = ['Control', 'DS', 'Abnormal']
    
    for test_subject in tqdm(subjects, desc=f"LOSO CV - {model_name}"):
        train_df = df[df['file_name'] != test_subject]
        test_df = df[df['file_name'] == test_subject]
        
        test_class = test_df['label'].iloc[0]
        test_class_name = class_names[test_class]
        
        X_train = np.array(train_df['data'].tolist())
        y_train = np.array(train_df['label'].tolist())
        X_test = np.array(test_df['data'].tolist())
        y_test = np.array(test_df['label'].tolist())
        
        scaler = StandardScaler()
        n_channels, n_times = X_train.shape[1], X_train.shape[2]
        X_train_flat = X_train.reshape(-1, n_channels * n_times)
        X_test_flat = X_test.reshape(-1, n_channels * n_times)
        
        X_train_norm = scaler.fit_transform(X_train_flat).reshape(-1, n_channels, n_times)
        X_test_norm = scaler.transform(X_test_flat).reshape(-1, n_channels, n_times)
        
        train_dataset = EEGDataset(X_train_norm, y_train, train_df['file_name'].values)
        test_dataset = EEGDataset(X_test_norm, y_test, test_df['file_name'].values)
        
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
        
        model = get_model(n_channels=8, n_classes=3, n_times=n_times, lstm_hidden=64)
        model = model.to(device)
        
      
        class_weights = torch.tensor([1.0, 3.25, 4.5]).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)

        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
        
        best_acc = 0
        patience_counter = 0
        max_patience = 20
        
        for epoch in range(EPOCHS):
            train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
            test_loss, test_acc, test_f1, test_labels, test_preds = evaluate(model, test_loader, criterion, device)
            
            scheduler.step(test_loss)
            
            if test_acc > best_acc:
                best_acc = test_acc
                best_f1 = test_f1
                best_y_true = test_labels
                best_y_pred = test_preds
                
                # Get probability scores
                model.eval()
                with torch.no_grad():
                    test_proba = []
                    for batch_x, _ in test_loader:
                        batch_x = batch_x.to(device)
                        outputs = model(batch_x)
                        probs = torch.softmax(outputs, dim=1)
                        test_proba.extend(probs.cpu().numpy())
                best_y_proba = np.array(test_proba)
                
                patience_counter = 0
            else:
                patience_counter += 1
            
            if patience_counter >= max_patience:
                break
        
        comp_metrics = calculate_metrics(best_y_true, best_y_pred, class_names)
        
        result = {
            'subject': test_subject,
            'subject_class': test_class_name,
            'model': model_name,
            'accuracy': best_acc,
            'balanced_accuracy': comp_metrics['balanced_accuracy'],
            'f1_weighted': comp_metrics['f1_weighted'],
            'f1_macro': comp_metrics['f1_macro'],
            'control_sensitivity': comp_metrics.get('Control_sensitivity', 0),
            'control_specificity': comp_metrics.get('Control_specificity', 0),
            'control_precision': comp_metrics.get('Control_precision', 0),
            'ds_sensitivity': comp_metrics.get('DS_sensitivity', 0),
            'ds_specificity': comp_metrics.get('DS_specificity', 0),
            'ds_precision': comp_metrics.get('DS_precision', 0),
            'abnormal_sensitivity': comp_metrics.get('Abnormal_sensitivity', 0),
            'abnormal_specificity': comp_metrics.get('Abnormal_specificity', 0),
            'abnormal_precision': comp_metrics.get('Abnormal_precision', 0),
            'samples': len(best_y_true)
        }
        
        results.append(result)
        pooled_y_true.extend(best_y_true)
        pooled_y_pred.extend(best_y_pred)
        pooled_y_proba.extend(best_y_proba)
        
        print(f"Subject {test_subject} ({test_class_name}): Acc={best_acc:.2f}%, F1={best_f1:.4f}")
        
        del model, train_loader, test_loader
        torch.cuda.empty_cache()
    
    results_df = pd.DataFrame(results)
    pooled_y_proba = np.array(pooled_y_proba)
    
    return results_df, pooled_y_true, pooled_y_pred, pooled_y_proba, class_names


# ============================================================
# STATISTICS AND VISUALIZATIONS
# ============================================================

def generate_statistics(results_df, pooled_y_true, pooled_y_pred, pooled_y_proba, class_names):
    """Generate comprehensive statistics"""
    
    print("\n" + "="*80)
    print("LOSO CV RESULTS - CUSTOM CNN-LSTM")
    print("="*80)
    
    # Overall metrics
    print("\n1. OVERALL PERFORMANCE METRICS (Pooled Predictions)")
    print("-" * 80)
    overall_metrics = calculate_metrics(pooled_y_true, pooled_y_pred, class_names)
    
    print(f"  Accuracy:           {overall_metrics['accuracy']:.4f}")
    print(f"  Balanced Accuracy:  {overall_metrics['balanced_accuracy']:.4f}")
    print(f"  F1 Score (weighted):{overall_metrics['f1_weighted']:.4f}")
    print(f"  F1 Score (macro):   {overall_metrics['f1_macro']:.4f}")
    
    # ROC-AUC
    print("\n2. ROC-AUC SCORES (One-vs-Rest)")
    print("-" * 80)
    roc_scores = calculate_roc_auc(pooled_y_true, pooled_y_proba, class_names)
    
    print(f"  Macro-Average:      {roc_scores['roc_auc_macro']:.4f}")
    print(f"  Weighted:           {roc_scores['roc_auc_weighted']:.4f}")
    print(f"  Control (OvR):      {roc_scores['roc_auc_Control']:.4f}")
    print(f"  DS (OvR):           {roc_scores['roc_auc_DS']:.4f}")
    print(f"  Abnormal (OvR):     {roc_scores['roc_auc_Abnormal']:.4f}")
    
    # Per-class metrics
    print("\n3. PER-CLASS METRICS (Pooled Predictions)")
    print("-" * 80)
    for class_name in class_names:
        print(f"\n  {class_name.upper()}:")
        print(f"    Sensitivity:    {overall_metrics[f'{class_name}_sensitivity']:.4f}")
        print(f"    Specificity:    {overall_metrics[f'{class_name}_specificity']:.4f}")
        print(f"    Precision:      {overall_metrics[f'{class_name}_precision']:.4f}")
        print(f"    NPV:            {overall_metrics[f'{class_name}_npv']:.4f}")
    
    # Pooled confusion matrix
    print("\n4. POOLED CONFUSION MATRIX")
    print("-" * 80)
    cm = confusion_matrix(pooled_y_true, pooled_y_pred, labels=[0, 1, 2])
    print(f"\n           Predicted")
    print(f"          Control    DS  Abnormal")
    for i, class_name in enumerate(class_names):
        print(f"  {class_name:10s} {cm[i, 0]:6d} {cm[i, 1]:5d} {cm[i, 2]:8d}")
    print(f"\n  Total Samples: {cm.sum()}")
    
    # Subject-level summary
    print("\n5. SUBJECT-LEVEL SUMMARY (Mean ± SD)")
    print("-" * 80)
    for class_name in class_names:
        class_data = results_df[results_df['subject_class'] == class_name]
        if len(class_data) > 0:
            print(f"\n  {class_name.upper()} (n={len(class_data)})")
            print(f"    Accuracy:    {class_data['accuracy'].mean():.2f}% ± {class_data['accuracy'].std():.2f}")
            print(f"    Bal. Acc:    {class_data['balanced_accuracy'].mean():.4f} ± {class_data['balanced_accuracy'].std():.4f}")
            print(f"    F1 (weighted):{class_data['f1_weighted'].mean():.4f} ± {class_data['f1_weighted'].std():.4f}")
    
    return overall_metrics, roc_scores, cm


def generate_tsne_plots(pooled_y_true, pooled_y_pred, pooled_y_proba, class_names, output_dir):
    """Generate t-SNE visualizations"""
    
    print("\n" + "="*80)
    print("GENERATING T-SNE VISUALIZATIONS")
    print("="*80)
    
    features = pooled_y_proba
    print(f"\nApplying t-SNE to {len(features)} samples...")
    
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(features)-1), n_iter=1000)
    features_tsne = tsne.fit_transform(features)
    
    print(f"t-SNE completed!")
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    color_map = {0: colors[0], 1: colors[1], 2: colors[2]}
    
    # Combined plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for class_idx, class_name in enumerate(class_names):
        mask = np.array(pooled_y_true) == class_idx
        axes[0].scatter(features_tsne[mask, 0], features_tsne[mask, 1],
                       c=color_map[class_idx], label=class_name, alpha=0.6, s=30, edgecolors='none')
    
    axes[0].set_xlabel('t-SNE Component 1', fontsize=12)
    axes[0].set_ylabel('t-SNE Component 2', fontsize=12)
    axes[0].set_title('t-SNE: True Labels', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    for class_idx, class_name in enumerate(class_names):
        mask = np.array(pooled_y_pred) == class_idx
        axes[1].scatter(features_tsne[mask, 0], features_tsne[mask, 1],
                       c=color_map[class_idx], label=class_name, alpha=0.6, s=30, edgecolors='none')
    
    axes[1].set_xlabel('t-SNE Component 1', fontsize=12)
    axes[1].set_ylabel('t-SNE Component 2', fontsize=12)
    axes[1].set_title('t-SNE: Predicted Labels', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plot_path = f"{output_dir}/TSNE_True_vs_Predicted.png"
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_path}")
    plt.close()
    
    # True labels
    fig, ax = plt.subplots(figsize=(10, 8))
    for class_idx, class_name in enumerate(class_names):
        mask = np.array(pooled_y_true) == class_idx
        ax.scatter(features_tsne[mask, 0], features_tsne[mask, 1],
                   c=color_map[class_idx], label=f'{class_name} (n={mask.sum()})',
                   alpha=0.7, s=40, edgecolors='black', linewidth=0.5)
    
    ax.set_xlabel('t-SNE Component 1', fontsize=13, fontweight='bold')
    ax.set_ylabel('t-SNE Component 2', fontsize=13, fontweight='bold')
    ax.set_title('t-SNE: True Labels', fontsize=15, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    
    plot_path = f"{output_dir}/TSNE_True_Labels.png"
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_path}")
    plt.close()
    
    # Predicted labels
    fig, ax = plt.subplots(figsize=(10, 8))
    for class_idx, class_name in enumerate(class_names):
        mask = np.array(pooled_y_pred) == class_idx
        ax.scatter(features_tsne[mask, 0], features_tsne[mask, 1],
                   c=color_map[class_idx], label=f'{class_name} (n={mask.sum()})',
                   alpha=0.7, s=40, edgecolors='black', linewidth=0.5)
    
    ax.set_xlabel('t-SNE Component 1', fontsize=13, fontweight='bold')
    ax.set_ylabel('t-SNE Component 2', fontsize=13, fontweight='bold')
    ax.set_title('t-SNE: Predicted Labels', fontsize=15, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    
    plot_path = f"{output_dir}/TSNE_Predicted_Labels.png"
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_path}")
    plt.close()
    
    # Save t-SNE data
    tsne_data = pd.DataFrame({
        'TSNE_1': features_tsne[:, 0],
        'TSNE_2': features_tsne[:, 1],
        'True_Label': pooled_y_true,
        'Predicted_Label': pooled_y_pred,
        'Prob_Control': pooled_y_proba[:, 0],
        'Prob_DS': pooled_y_proba[:, 1],
        'Prob_Abnormal': pooled_y_proba[:, 2],
    })
    
    tsne_data.to_csv(f"{output_dir}/TSNE_Data.csv", index=False)
    print(f"✓ Saved: TSNE_Data.csv")


def plot_confusion_matrix(cm, class_names, output_dir):
    """Plot confusion matrix"""
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, 
                yticklabels=class_names, ax=ax, cbar=True)
    ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
    ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plot_path = f"{output_dir}/Confusion_Matrix.png"
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_path}")
    plt.close()


def create_output_tables(results_df, pooled_y_true, pooled_y_pred, pooled_y_proba, class_names, output_dir):
    """Create CSV tables for paper"""
    
    print("\n" + "="*80)
    print("CREATING OUTPUT TABLES")
    print("="*80)
    
    # Subject-level results
    table1 = results_df[['subject', 'subject_class', 'accuracy', 'balanced_accuracy', 
                          'f1_weighted', 'f1_macro']].copy()
    table1.to_csv(f"{output_dir}/Table_Subject_Results.csv", index=False)
    print(f"✓ Table_Subject_Results.csv")
    
    # Pooled metrics
    overall_metrics = calculate_metrics(pooled_y_true, pooled_y_pred, class_names)
    roc_scores = calculate_roc_auc(pooled_y_true, pooled_y_proba, class_names)
    
    table2_data = {
        'Metric': [
            'Accuracy', 'Balanced Accuracy', 'F1 (Weighted)', 'F1 (Macro)',
            'ROC-AUC (Macro)', 'ROC-AUC (Weighted)',
            'Control_Sensitivity', 'Control_Specificity', 'Control_Precision', 'Control_NPV',
            'DS_Sensitivity', 'DS_Specificity', 'DS_Precision', 'DS_NPV',
            'Abnormal_Sensitivity', 'Abnormal_Specificity', 'Abnormal_Precision', 'Abnormal_NPV',
            'ROC-AUC_Control', 'ROC-AUC_DS', 'ROC-AUC_Abnormal'
        ],
        'Value': [
            f"{overall_metrics['accuracy']:.4f}",
            f"{overall_metrics['balanced_accuracy']:.4f}",
            f"{overall_metrics['f1_weighted']:.4f}",
            f"{overall_metrics['f1_macro']:.4f}",
            f"{roc_scores['roc_auc_macro']:.4f}",
            f"{roc_scores['roc_auc_weighted']:.4f}",
            f"{overall_metrics['Control_sensitivity']:.4f}",
            f"{overall_metrics['Control_specificity']:.4f}",
            f"{overall_metrics['Control_precision']:.4f}",
            f"{overall_metrics['Control_npv']:.4f}",
            f"{overall_metrics['DS_sensitivity']:.4f}",
            f"{overall_metrics['DS_specificity']:.4f}",
            f"{overall_metrics['DS_precision']:.4f}",
            f"{overall_metrics['DS_npv']:.4f}",
            f"{overall_metrics['Abnormal_sensitivity']:.4f}",
            f"{overall_metrics['Abnormal_specificity']:.4f}",
            f"{overall_metrics['Abnormal_precision']:.4f}",
            f"{overall_metrics['Abnormal_npv']:.4f}",
            f"{roc_scores['roc_auc_Control']:.4f}",
            f"{roc_scores['roc_auc_DS']:.4f}",
            f"{roc_scores['roc_auc_Abnormal']:.4f}",
        ]
    }
    
    table2 = pd.DataFrame(table2_data)
    table2.to_csv(f"{output_dir}/Table_Overall_Metrics.csv", index=False)
    print(f"✓ Table_Overall_Metrics.csv")
    
    # Confusion matrix
    cm = confusion_matrix(pooled_y_true, pooled_y_pred, labels=[0, 1, 2])
    table3_data = {
        'Actual_Class': class_names,
        'Pred_Control': cm[:, 0],
        'Pred_DS': cm[:, 1],
        'Pred_Abnormal': cm[:, 2],
        'Total': cm.sum(axis=1)
    }
    
    table3 = pd.DataFrame(table3_data)
    table3.to_csv(f"{output_dir}/Table_Confusion_Matrix.csv", index=False)
    print(f"✓ Table_Confusion_Matrix.csv")
    
    # Pooled predictions
    pooled_df = pd.DataFrame({
        'True_Label': pooled_y_true,
        'Predicted_Label': pooled_y_pred,
        'Prob_Control': pooled_y_proba[:, 0],
        'Prob_DS': pooled_y_proba[:, 1],
        'Prob_Abnormal': pooled_y_proba[:, 2],
        'Confidence': np.max(pooled_y_proba, axis=1)
    })
    
    pooled_df.to_csv(f"{output_dir}/Pooled_Predictions.csv", index=False)
    print(f"✓ Pooled_Predictions.csv")


# ============================================================
# MAIN PIPELINE
# ============================================================

if __name__ == "__main__":
    print("="*80)
    print("EEG THREE-CLASS CLASSIFICATION - LOSO CV")
    print("Model: Custom CNN-LSTM (Non-Pretrained)")
    print("Classes: Control (0), DS (1), Abnormal (2)")
    print("="*80)

    print("\n1. LOADING EEG DATA")
    print("-" * 80)

    print("\n   Loading Control EEG...")
    ctrl_chunks, ctrl_lbl, ctrl_files = load_eeg_multi_folders(
        [CONTROL_FOLDER, CONTROL_FOLDER_2], label=0, fs=FS, window_sec=WINDOW_SEC, 
        overlap=OVERLAP, remove_artifacts=True    )
    print(f"   ✓ Control: {len(ctrl_chunks)} samples from {len(set(ctrl_files))} subjects")

    print("\n   Loading DS EEG...")
    ds_chunks, ds_lbl, ds_files = load_eeg_multi_folders(
        [DS_FOLDER, DS_FOLDER_2], label=1, fs=FS, window_sec=WINDOW_SEC, 
        overlap=OVERLAP, remove_artifacts=True    )
    print(f"   ✓ DS: {len(ds_chunks)} samples from {len(set(ds_files))} subjects")

    print("\n   Loading Abnormal EEG...")
    abnormal_chunks, abnormal_lbl, abnormal_files = load_eeg_folder(
        ABNORMAL_FOLDER, 2, FS, WINDOW_SEC, OVERLAP, remove_artifacts=True   )
    print(f"   ✓ Abnormal: {len(abnormal_chunks)} samples from {len(set(abnormal_files))} subjects")

    df_ctrl = pd.DataFrame({'file_name': ctrl_files, 'label': ctrl_lbl, 'data': ctrl_chunks})
    df_ds = pd.DataFrame({'file_name': ds_files, 'label': ds_lbl, 'data': ds_chunks})
    df_abnormal = pd.DataFrame({'file_name': abnormal_files, 'label': abnormal_lbl, 'data': abnormal_chunks})

    df_combined = pd.concat([df_ctrl, df_ds, df_abnormal], ignore_index=True)

    print(f"\n{'='*80}")
    print("DATASET SUMMARY")
    print(f"{'='*80}")
    print(f"Total samples: {len(df_combined)}")
    print(f"Total subjects: {df_combined['file_name'].nunique()}")
    print(f"\nClass distribution:")
    class_dist = df_combined['label'].value_counts().sort_index()
    subject_dist = df_combined.groupby('label')['file_name'].nunique()
    print(f"  Control (0): {class_dist.get(0, 0)} samples, {subject_dist.get(0, 0)} subjects")
    print(f"  DS (1): {class_dist.get(1, 0)} samples, {subject_dist.get(1, 0)} subjects")
    print(f"  Abnormal (2): {class_dist.get(2, 0)} samples, {subject_dist.get(2, 0)} subjects")

    # Run LOSO CV
    print(f"\n{'='*80}")
    print("2. RUNNING LOSO CV WITH CUSTOM CNN-LSTM")
    print(f"{'='*80}")

    results_df, pooled_y_true, pooled_y_pred, pooled_y_proba, class_names = run_loso_cv(df_combined, 'CustomCNNLSTM')

    # Generate statistics
    print("\n3. GENERATING STATISTICS")
    overall_metrics, roc_scores, cm = generate_statistics(results_df, pooled_y_true, pooled_y_pred, pooled_y_proba, class_names)

    # Generate visualizations
    print("\n4. GENERATING VISUALIZATIONS")
    generate_tsne_plots(pooled_y_true, pooled_y_pred, pooled_y_proba, class_names, OUTPUT_DIR)
    plot_confusion_matrix(cm, class_names, OUTPUT_DIR)

    # Create output tables
    print("\n5. CREATING OUTPUT TABLES")
    create_output_tables(results_df, pooled_y_true, pooled_y_pred, pooled_y_proba, class_names, OUTPUT_DIR)

    # Save results
    results_df.to_csv(f"{OUTPUT_DIR}/LOSO_Subject_Results.csv", index=False)

    print(f"\n{'='*80}")
    print("✓ PIPELINE COMPLETED SUCCESSFULLY")
    print(f"{'='*80}")
    print(f"\nResults saved to: {OUTPUT_DIR}")
    print(f"\nGenerated files:")
    print(f"  📊 Tables:")
    print(f"     - Table_Subject_Results.csv")
    print(f"     - Table_Overall_Metrics.csv")
    print(f"     - Table_Confusion_Matrix.csv")
    print(f"     - Pooled_Predictions.csv")
    print(f"     - LOSO_Subject_Results.csv")
    print(f"  📈 Visualizations:")
    print(f"     - TSNE_True_vs_Predicted.png")
    print(f"     - TSNE_True_Labels.png")
    print(f"     - TSNE_Predicted_Labels.png")
    print(f"     - Confusion_Matrix.png")
    print(f"     - TSNE_Data.csv")
    print(f"\n✓ Ready for paper submission!")